In [8]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import os

DRIVE_PATH = '/content/drive/MyDrive/data/raw'
LOCAL_PATH = '/content/data/raw'

os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(LOCAL_PATH, exist_ok=True)

print("✅ Drive monté et dossiers prêts")
print("Fichiers disponibles sur Drive :")
for f in os.listdir(DRIVE_PATH):
    print(f"  {f}")

Mounted at /content/drive
✅ Drive monté et dossiers prêts
Fichiers disponibles sur Drive :
  benin_gkg.csv
  benin_media_bias.csv
  benin_events_clean.csv
  comparatif_regional.csv
  benin_eco_events.csv
  benin_bilateral.csv
  benin_sector_themes.csv
  fmi_comparatif.csv


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style global
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.family']    = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
sns.set_theme(style='whitegrid', palette='muted')

# ── Chargement ─────────────────────────────────────────────────────────────
df_events    = pd.read_csv(f'{DRIVE_PATH}/benin_events_clean.csv')
df_regional  = pd.read_csv(f'{DRIVE_PATH}/comparatif_regional.csv')
df_eco       = pd.read_csv(f'{DRIVE_PATH}/benin_eco_events.csv')
df_bilateral = pd.read_csv(f'{DRIVE_PATH}/benin_bilateral.csv')
df_bias      = pd.read_csv(f'{DRIVE_PATH}/benin_media_bias.csv')
df_themes    = pd.read_csv(f'{DRIVE_PATH}/benin_sector_themes.csv')
df_fmi       = pd.read_csv(f'{DRIVE_PATH}/fmi_comparatif.csv')

# ── Conversions dates ──────────────────────────────────────────────────────
df_events['date']    = pd.to_datetime(df_events['SQLDATE'].astype(str), format='%Y%m%d')
df_events['mois']    = df_events['date'].dt.to_period('M').astype(str)
df_events['semaine'] = df_events['date'].dt.to_period('W').astype(str)

# ── Résumé ─────────────────────────────────────────────────────────────────
print("=== DATASETS CHARGÉS ===\n")
for nom, df in [
    ('benin_events_clean',  df_events),
    ('comparatif_regional', df_regional),
    ('benin_eco_events',    df_eco),
    ('benin_bilateral',     df_bilateral),
    ('benin_media_bias',    df_bias),
    ('benin_sector_themes', df_themes),
    ('fmi_comparatif',      df_fmi),
]:
    print(f"  {nom:<25} {len(df):>6,} lignes  |  {df.shape[1]} colonnes")

=== DATASETS CHARGÉS ===

  benin_events_clean        25,629 lignes  |  33 colonnes
  comparatif_regional           91 lignes  |  14 colonnes
  benin_eco_events          10,079 lignes  |  19 colonnes
  benin_bilateral              104 lignes  |  10 colonnes
  benin_media_bias             151 lignes  |  8 colonnes
  benin_sector_themes          117 lignes  |  4 colonnes
  fmi_comparatif                63 lignes  |  9 colonnes


In [1]:
# ==============================================================
# BENIN PULSE — Script d'extraction des données médias
# Couvre : sources WordPress, RSS, et scraping HTML direct
# ==============================================================
 
 
# ==============================================================
# Installation des dépendances
# ==============================================================
 
import subprocess
subprocess.run(
    ["pip", "install", "requests", "beautifulsoup4", "pandas",
     "feedparser", "lxml", "tqdm", "-q"],
    check=True
)


CompletedProcess(args=['pip', 'install', 'requests', 'beautifulsoup4', 'pandas', 'feedparser', 'lxml', 'tqdm', '-q'], returncode=0)

In [24]:
# ============================================================
# Montage Google Drive et dossier de travail
# ============================================================

import os
from google.colab import drive

drive.mount("/content/drive")

WORKDIR = "/content/drive/MyDrive/Benin_Pulse"
os.chdir(WORKDIR)
print(f"Dossier de travail : {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dossier de travail : /content/drive/MyDrive/Benin_Pulse


In [3]:
# ==============================================================
# Imports et configuration globale
# ==============================================================
 
import os
import time
import logging
import warnings
import requests
import feedparser
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from tqdm.notebook import tqdm
 
# Supprime le warning BS4 déclenché quand un champ contient une URL brute
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("benin_pulse")
 
os.chdir("/content/drive/MyDrive/Benin_Pulse")
 
DATE_DEBUT   = "2025-01-01"
DATE_FIN     = "2026-05-31"
OUTPUT_FILE  = "benin_raw_media.csv"
DELAI_SOURCE = 2
TIMEOUT      = 15
 
# Headers plus crédibles pour contourner les protections anti-bot (403)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "application/json, text/plain, */*",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
}
 

In [4]:
# ==============================================================
# Sources WordPress
# ==============================================================

SOURCES_WORDPRESS = [
    # Confirmees actives sur 2 executions consecutives depuis Colab
    {"nom": "La Nouvelle Tribune", "url": "https://lanouvelletribune.info"},
    {"nom": "Matin Libre",         "url": "https://matinlibre.com"},
    {"nom": "Le Matinal",          "url": "https://lematinal.bj"},
    {"nom": "Ecobenin",            "url": "https://ecobenin.bj"},
    {"nom": "Togobreakingnews",    "url": "https://togobreakingnews.com"},
]
 
 
def _parse_date(date_str):
    """Convertit une date ISO WordPress en objet datetime. Retourne None si invalide."""
    try:
        return datetime.strptime(date_str.split("T")[0], "%Y-%m-%d")
    except (ValueError, AttributeError):
        return None
 
 
def collecter_wordpress(source):
    """
    Collecte les articles d'une source WordPress via l'API REST.
    Pagine jusqu'à X-WP-TotalPages ou jusqu'à dépasser DATE_DEBUT.
    """
    articles   = []
    page       = 1
    date_min   = datetime.strptime(DATE_DEBUT, "%Y-%m-%d")
    date_max   = datetime.strptime(DATE_FIN,   "%Y-%m-%d")
    nom        = source["nom"]
    base_url   = source["url"].rstrip("/")
 
    while True:
        endpoint = (
            f"{base_url}/wp-json/wp/v2/posts"
            f"?page={page}&per_page=50&orderby=date&order=desc"
        )
        try:
            resp = requests.get(endpoint, headers=HEADERS, timeout=TIMEOUT)
        except requests.exceptions.RequestException as exc:
            log.warning("%s — erreur réseau page %d : %s", nom, page, exc)
            break
 
        if resp.status_code == 400:
            break
        if resp.status_code != 200:
            log.warning("%s — HTTP %d, arrêt.", nom, resp.status_code)
            break
 
        total_pages = int(resp.headers.get("X-WP-TotalPages", 1))
 
        try:
            posts = resp.json()
        except ValueError:
            log.warning("%s — réponse JSON invalide page %d.", nom, page)
            break
 
        if not posts:
            break
 
        stop = False
        for post in posts:
            date_article = _parse_date(post.get("date", ""))
            if date_article is None:
                continue
            if date_article > date_max:
                continue
            if date_article < date_min:
                stop = True
                break
 
            titre = BeautifulSoup(
                post.get("title",  {}).get("rendered", ""), "lxml"
            ).get_text().strip()
 
            resume = BeautifulSoup(
                post.get("excerpt", {}).get("rendered", ""), "lxml"
            ).get_text().strip()[:800]
 
            articles.append({
                "date":   date_article.strftime("%Y-%m-%d"),
                "source": nom,
                "type":   "wordpress",
                "titre":  titre,
                "resume": resume,
                "url":    post.get("link", ""),
            })
 
        if stop or page >= total_pages:
            break
 
        page += 1
 
    log.info("%s — %d articles collectés.", nom, len(articles))
    return articles

In [5]:
# ==============================================================
# Sources RSS
# ==============================================================

SOURCES_RSS = [
    # ----------------------------------------------------------
    # SOURCES DEDIEES BENIN — pas de filtre mots-cles
    # ----------------------------------------------------------
    {
        "nom": "RFI Benin",
        "url": "https://www.rfi.fr/fr/tag/b%C3%A9nin/rss",
        "dedie": True,
    },
    {
        "nom": "24 Heures au Benin",
        "url": "https://www.24haubenin.info/feed/",
        "dedie": True,
    },
    {
        "nom": "Banouto",
        "url": "https://www.banouto.bj/feed/",
        "dedie": True,
    },
    {
        "nom": "La Nation Benin",
        "url": "https://lanation.bj/feed/",
        "dedie": True,
    },
    {
        "nom": "Gouvernement du Benin",
        "url": "https://gouv.bj/feed/",
        "dedie": True,
    },
    {
        "nom": "aCotonou",
        "url": "http://news.acotonou.com/rss.xml",
        "dedie": True,
    },
    {
        "nom": "BCEAO Communiques",
        "url": "https://www.bceao.int/fr/rss.xml",
        "dedie": True,
    },
    # ----------------------------------------------------------
    # SOURCES INTERNATIONALES — filtrage mots-cles Benin actif
    # ----------------------------------------------------------
    {
        "nom": "Le Monde Afrique",
        "url": "https://www.lemonde.fr/afrique/rss_full.xml",
        "dedie": False,
    },
    {
        "nom": "Jeune Afrique",
        "url": "https://www.jeuneafrique.com/feed/",
        "dedie": False,
    },
    {
        "nom": "BBC Afrique",
        "url": "https://feeds.bbci.co.uk/afrique/rss.xml",
        "dedie": False,
    },
    {
        "nom": "Agence Ecofin",
        "url": "https://www.agenceecofin.com/rss/all",
        "dedie": False,
    },
    {
        "nom": "Africa Intelligence",
        "url": "https://www.africaintelligence.fr/rss",
        "dedie": False,
    },
]

# Sources dont le flux est entierement dedie au Benin
SOURCES_RSS_DEDIES = {s["nom"] for s in SOURCES_RSS if s["dedie"]}

KEYWORDS_BENIN = [
    "bénin", "benin", "cotonou", "porto-novo",
    "patrice talon", "wadagni", "gdiz", "ouidah",
    "abomey", "parakou", "djigbe",
]


def _est_pertinent_benin(titre, resume):
    texte = (titre + " " + resume).lower()
    return any(kw in texte for kw in KEYWORDS_BENIN)


def collecter_rss(source, filtrer_benin=False):
    articles = []
    date_min = datetime.strptime(DATE_DEBUT, "%Y-%m-%d")
    date_max = datetime.strptime(DATE_FIN,   "%Y-%m-%d")
    nom      = source["nom"]

    try:
        feed = feedparser.parse(source["url"])
    except Exception as exc:
        log.warning("%s — echec parsing RSS : %s", nom, exc)
        return articles

    for entry in feed.entries:
        date_article = None
        for attr in ("published_parsed", "updated_parsed"):
            t = getattr(entry, attr, None)
            if t:
                try:
                    date_article = datetime(*t[:6])
                    break
                except (TypeError, ValueError):
                    continue

        if date_article is None:
            continue
        if not (date_min <= date_article <= date_max):
            continue

        titre  = entry.get("title", "").strip()
        resume = BeautifulSoup(
            entry.get("summary", ""), "lxml"
        ).get_text().strip()[:800]

        if filtrer_benin and not _est_pertinent_benin(titre, resume):
            continue

        articles.append({
            "date":   date_article.strftime("%Y-%m-%d"),
            "source": nom,
            "type":   "rss",
            "titre":  titre,
            "resume": resume,
            "url":    entry.get("link", ""),
        })

    log.info("%s — %d articles collectes.", nom, len(articles))
    return articles

In [6]:
# ==============================================================
# Sources HTML (scraping direct)
# ==============================================================
 
# Pour les sites sans API ni RSS, on parse le HTML de la page
# des actualités et on suit chaque lien d'article.
 
SOURCES_HTML = [
    {
        "nom":            "Gouvernement du Bénin",
        "url_liste":      "https://gouv.bj/actualites/",
        "selecteur_lien": "article a",
        "selecteur_date": "time",
        "selecteur_titre":"h1, h2.entry-title",
        "selecteur_corps":"div.entry-content, div.article-content",
        "base_url":       "https://gouv.bj",
    },
    {
        "nom":            "API-BJ (Investissements)",
        "url_liste":      "https://investinbenin.com/actualites/",
        "selecteur_lien": "article a",
        "selecteur_date": "time",
        "selecteur_titre":"h1",
        "selecteur_corps":"div.entry-content",
        "base_url":       "https://investinbenin.com",
    },
]
 
 
def _extraire_article_html(url, cfg):
    """Visite un article et extrait titre, date et résumé."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, "lxml")
 
        # Date
        tag_date = soup.select_one(cfg["selecteur_date"])
        date_str = None
        if tag_date:
            date_str = tag_date.get("datetime", tag_date.get_text())
        date_article = _parse_date(date_str) if date_str else None
 
        # Titre
        tag_titre = soup.select_one(cfg["selecteur_titre"])
        titre = tag_titre.get_text().strip() if tag_titre else ""
 
        # Corps (résumé = 800 premiers caractères)
        tag_corps = soup.select_one(cfg["selecteur_corps"])
        resume = tag_corps.get_text(" ", strip=True)[:800] if tag_corps else ""
 
        return {"date": date_article, "titre": titre, "resume": resume}
    except Exception as exc:
        log.debug("Erreur extraction %s : %s", url, exc)
        return None
 
 
def collecter_html(source):
    """
    Scrape la page de liste d'actualités d'une source HTML,
    puis visite chaque article individuel.
    """
    articles = []
    date_min = datetime.strptime(DATE_DEBUT, "%Y-%m-%d")
    date_max = datetime.strptime(DATE_FIN,   "%Y-%m-%d")
    nom      = source["nom"]
 
    try:
        resp = requests.get(source["url_liste"], headers=HEADERS, timeout=TIMEOUT)
        soup = BeautifulSoup(resp.text, "lxml")
    except Exception as exc:
        log.warning("%s — impossible de charger la page liste : %s", nom, exc)
        return articles
 
    liens = []
    for tag in soup.select(source["selecteur_lien"]):
        href = tag.get("href", "")
        if href.startswith("http"):
            liens.append(href)
        elif href.startswith("/"):
            liens.append(source["base_url"] + href)
 
    liens = list(dict.fromkeys(liens))  # dédoublonnage ordre préservé
    log.info("%s — %d liens trouvés.", nom, len(liens))
 
    for lien in liens:
        data = _extraire_article_html(lien, source)
        if not data:
            continue
        if data["date"] is None:
            continue
        if not (date_min <= data["date"] <= date_max):
            continue
 
        articles.append({
            "date":   data["date"].strftime("%Y-%m-%d"),
            "source": nom,
            "type":   "html",
            "titre":  data["titre"],
            "resume": data["resume"],
            "url":    lien,
        })
        time.sleep(0.5)  # pause polie entre chaque article
 
    log.info("%s — %d articles collectés.", nom, len(articles))
    return articles
 

In [7]:
# ==============================================================
# Orchestration
# ==============================================================

tous_articles = []

print("=" * 60)
print("COLLECTE WORDPRESS")
print("=" * 60)
for source in tqdm(SOURCES_WORDPRESS, desc="WordPress"):
    tous_articles.extend(collecter_wordpress(source))
    time.sleep(DELAI_SOURCE)

print()
print("=" * 60)
print("COLLECTE RSS")
print("=" * 60)
for source in tqdm(SOURCES_RSS, desc="RSS"):
    filtrer = source["nom"] not in SOURCES_RSS_DEDIES
    tous_articles.extend(collecter_rss(source, filtrer_benin=filtrer))
    time.sleep(DELAI_SOURCE)

print()
print("=" * 60)
print("COLLECTE HTML DIRECT")
print("=" * 60)
for source in tqdm(SOURCES_HTML, desc="HTML"):
    tous_articles.extend(collecter_html(source))
    time.sleep(DELAI_SOURCE)


COLLECTE WORDPRESS


WordPress:   0%|          | 0/5 [00:00<?, ?it/s]


COLLECTE RSS


RSS:   0%|          | 0/12 [00:00<?, ?it/s]


COLLECTE HTML DIRECT


HTML:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
# ==============================================================
# Nettoyage, déduplication et sauvegarde
# ==============================================================
 
df = pd.DataFrame(tous_articles)
 
n_brut = len(df)
 
# Supprimer les doublons exacts (même URL)
df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
 
# Supprimer les doublons de contenu (même titre + même source)
df = df.drop_duplicates(subset=["titre", "source"]).reset_index(drop=True)
 
# Écarter les lignes sans titre ni résumé
df = df[
    df["titre"].str.strip().str.len() > 3
].reset_index(drop=True)
 
# Trier par date décroissante
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date", ascending=False).reset_index(drop=True)
df["date"] = df["date"].dt.strftime("%Y-%m-%d")
 
n_final = len(df)
 
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
 
print()
print("=" * 60)
print(f"Collecte terminée.")
print(f"  Articles bruts collectés : {n_brut}")
print(f"  Doublons supprimés       : {n_brut - n_final}")
print(f"  Articles conservés       : {n_final}")
print(f"  Fichier de sortie        : {OUTPUT_FILE}")
print("=" * 60)
print()
 
# Distribution par source
print("Distribution par source :")
dist = (
    df.groupby(["source", "type"])
    .size()
    .reset_index(name="articles")
    .sort_values("articles", ascending=False)
)
display(dist)



Collecte terminée.
  Articles bruts collectés : 32522
  Doublons supprimés       : 300
  Articles conservés       : 32222
  Fichier de sortie        : benin_raw_media.csv

Distribution par source :


,source,type,articles
3,La Nouvelle Tribune,wordpress,17339
4,Le Matinal,wordpress,8669
5,Matin Libre,wordpress,6081
1,Ecobenin,wordpress,97
6,RFI Benin,rss,24
0,BCEAO Communiques,rss,10
2,Jeune Afrique,rss,2


In [9]:
# ============================================================
# Installation des dépendances
# ============================================================

import subprocess
subprocess.run([
    "pip", "install", "transformers", "pandas", "tqdm", "torch", "-q"
], check=True)

CompletedProcess(args=['pip', 'install', 'transformers', 'pandas', 'tqdm', 'torch', '-q'], returncode=0)

In [10]:
# ============================================================
# Vérification du matériel (GPU / CPU)
# ============================================================

import torch

device = 0 if torch.cuda.is_available() else -1
device_label = torch.cuda.get_device_name(0) if device == 0 else "CPU"

print(f"Device : {device_label}")

if device == -1:
    print("Attention : aucun GPU detecte. Le traitement sur CPU sera significativement plus lent.")

Device : Tesla T4


In [11]:
# ============================================================
# Imports et configuration globale
# ============================================================

import pandas as pd
from transformers import pipeline
from tqdm.notebook import tqdm

# Seuil de confiance minimal pour retenir un theme
CONFIDENCE_THRESHOLD = 0.50

# Fichiers d'entree et de sortie
INPUT_FILE  = "benin_raw_media.csv"
OUTPUT_FILE = "benin_pulse_donnees_propres.csv"

# Taxonomie thematique Benin Pulse
THEMES = [
    "diaspora_retour",
    "vie_quotidienne",
    "droits_libertes",
    "sante_education",
    "culture_identite",
    "opportunites_investissement",
    "croissance_economique",
    "gouvernance_institutionnelle",
    "fiscalite_reglementation",
    "relations_internationales",
    "creation_entreprise",
    "foncier_immobilier",
    "agriculture_agrobusiness",
    "industrie_GDIZ",
    "numerique_innovation",
    "infrastructure_energie",
    "securite_stabilite",
    "tourisme",
    "vodun_culture_religieuse",
    "environnement_climat",
]

In [12]:
import logging
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

In [13]:
# ============================================================
# Initialisation du classificateur zero-shot
# ============================================================

# xlm-roberta-large-xnli : robuste sur le francais et les langues
# locales (fon, yoruba) potentiellement presentes dans les textes
classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=device,
    batch_size=16,  # optimal pour un GPU T4 (16 Go VRAM)
)

print("Modele charge en memoire.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Modele charge en memoire.


In [14]:
# ============================================================
# Chargement et preparation des donnees
# ============================================================

df = pd.read_csv(INPUT_FILE)
print(f"{len(df)} articles charges depuis '{INPUT_FILE}'.")

# Concatener titre et resume pour maximiser le contexte semantique.
# Les valeurs manquantes sont remplacees par une chaine vide,
# puis les lignes sans contenu exploitable sont ecartees.
df["texte_analyse"] = (
    df["titre"].fillna("") + ". " + df["resume"].fillna("")
).str.strip()

n_before = len(df)
df = df[df["texte_analyse"].str.len() > 10].reset_index(drop=True)
n_dropped = n_before - len(df)

if n_dropped > 0:
    print(f"{n_dropped} lignes ecartees (titre et resume vides).")

textes = df["texte_analyse"].tolist()
print(f"{len(textes)} articles prets pour l'inference.")

32222 articles charges depuis 'benin_raw_media.csv'.
32222 articles prets pour l'inference.


In [19]:
# ============================================================
# Inference sur echantillon representatif — 5000 articles
# ============================================================

import random
from torch.utils.data import Dataset

random.seed(42)
indices_sample = random.sample(range(len(textes)), 2000)
textes_sample  = [textes[i] for i in indices_sample]
df_sample      = df.iloc[indices_sample].reset_index(drop=True)

print(f"Echantillon : {len(textes_sample)} articles selectes.")

class ListDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        return self.data[i]

dataset  = ListDataset(textes_sample)
resultats = []
barre    = tqdm(total=len(textes_sample), desc="Classification")

for output in classifier(
    dataset,
    candidate_labels=THEMES,
    multi_label=True,
    batch_size=64,
    truncation=True,
    max_length=128,
):
    resultats.append(output)
    barre.update(1)

barre.close()
print(f"Inference terminee — {len(resultats)} articles traites.")

Echantillon : 2000 articles selectes.


Classification:   0%|          | 0/2000 [00:00<?, ?it/s]

Inference terminee — 2000 articles traites.


In [21]:
# ============================================================
# Post-traitement — sur l'echantillon uniquement
# ============================================================

themes_principaux = []
themes_multiples  = []

for result in resultats:
    labels_valides = [
        label
        for label, score in zip(result["labels"], result["scores"])
        if score > CONFIDENCE_THRESHOLD
    ]

    if not labels_valides:
        themes_principaux.append("general")
        themes_multiples.append(["general"])
    else:
        themes_principaux.append(labels_valides[0])
        themes_multiples.append(labels_valides)

# Travailler sur df_sample, pas df
df_sample["theme_principal"] = themes_principaux
df_sample["tags_secondaires"] = [
    ", ".join(tags[1:]) if len(tags) > 1 else ""
    for tags in themes_multiples
]
df_sample["tous_les_tags"] = [
    ", ".join(tags) for tags in themes_multiples
]

print(f"Post-traitement termine — {len(df_sample)} articles enrichis.")

Post-traitement termine — 2000 articles enrichis.


In [22]:
# ============================================================
# Sauvegarde de l'echantillon enrichi
# ============================================================

OUTPUT_FILE = "benin_pulse_donnees_propres.csv"
df_sample.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Fichier sauvegarde : {OUTPUT_FILE} ({len(df_sample)} lignes).")

# Distribution des themes
distribution = (
    df_sample["theme_principal"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "theme", "theme_principal": "count"})
)
print("\nDistribution des themes :")
display(distribution)

display(df_sample[["titre", "theme_principal", "tags_secondaires"]].head())

Fichier sauvegarde : benin_pulse_donnees_propres.csv (2000 lignes).

Distribution des themes :


,count,count
0,gouvernance_institutionnelle,366
1,relations_internationales,287
2,general,160
3,numerique_innovation,129
4,opportunites_investissement,127
5,agriculture_agrobusiness,113
6,diaspora_retour,110
7,creation_entreprise,100
8,fiscalite_reglementation,83
9,vodun_culture_religieuse,60


,titre,theme_principal,tags_secondaires
0,Grand Prix Littéraire du Bénin Edition 2025: L...,tourisme,diaspora_retour
1,Pour falsification d’une quittance de 40 000 F...,fiscalite_reglementation,"foncier_immobilier, infrastructure_energie, ag..."
2,Bénin/ Gdiz : Une révolution industrielle qui ...,industrie_GDIZ,"croissance_economique, creation_entreprise, nu..."
3,Pour « fautes commises » : Alain Adihou et cin...,gouvernance_institutionnelle,"diaspora_retour, vie_quotidienne, agriculture_..."
4,Communales et législatives de 2026: Sacca Lafi...,foncier_immobilier,"agriculture_agrobusiness, gouvernance_institut..."


In [25]:
# ============================================================
# Chargement unifie de tous les datasets
# ============================================================

import pandas as pd
import os

# Les deux emplacements
PATH_DATA    = "/content/drive/MyDrive/data/raw"
PATH_PULSE   = "/content/drive/MyDrive/Benin_Pulse"

FICHIERS = {
    # Dataset          : chemin
    "events"    : f"{PATH_DATA}/benin_events_clean.csv",
    "regional"  : f"{PATH_DATA}/comparatif_regional.csv",
    "eco"       : f"{PATH_DATA}/benin_eco_events.csv",
    "bilateral" : f"{PATH_DATA}/benin_bilateral.csv",
    "bias"      : f"{PATH_DATA}/benin_media_bias.csv",
    "secteurs"  : f"{PATH_DATA}/benin_sector_themes.csv",
    "fmi"       : f"{PATH_DATA}/fmi_comparatif.csv",
    "local"     : f"{PATH_PULSE}/benin_pulse_donnees_propres.csv",
}

datasets = {}
print(f"{'Dataset':<12} {'Statut':<10} {'Lignes':>7}  Colonnes")
print("-" * 75)

for nom, chemin in FICHIERS.items():
    try:
        df_tmp = pd.read_csv(chemin)
        datasets[nom] = df_tmp
        cols = ", ".join(df_tmp.columns[:4])
        print(f"{nom:<12} {'OK':<10} {len(df_tmp):>7}  {cols}")
    except FileNotFoundError:
        print(f"{nom:<12} {'INTROUVABLE':<10}")
    except Exception as e:
        print(f"{nom:<12} {'ERREUR':<10} {str(e)[:40]}")

print(f"\n{len(datasets)}/8 datasets charges.")

Dataset      Statut      Lignes  Colonnes
---------------------------------------------------------------------------
events       OK           25629  GLOBALEVENTID, SQLDATE, DATEADDED, QuadClass
regional     OK              91  fips_pays, annee, mois, nb_evenements
eco          OK           10079  GLOBALEVENTID, SQLDATE, Actor1Name, Actor1CountryCode
bilateral    OK             104  annee, mois, pays_partenaire, nb_interactions
bias         OK             151  semaine, source_type, periode, nb_evenements
secteurs     OK             117  mois, secteur, nb_mentions, tone_moyen
fmi          OK              63  pays_code, pays_nom, annee, balance_courante_pib
local        OK            2000  date, source, type, titre

8/8 datasets charges.


In [26]:
for nom in ["secteurs", "bias", "local", "events", "regional"]:
    print(f"\n=== {nom} ===")
    print(datasets[nom].columns.tolist())
    print(datasets[nom].head(3).to_string())


=== secteurs ===
['mois', 'secteur', 'nb_mentions', 'tone_moyen']
     mois    secteur  nb_mentions  tone_moyen
0  202505   Economie          215    0.559181
1  202505      Sante          150   -0.161127
2  202505  Politique          144    0.010774

=== bias ===
['semaine', 'source_type', 'periode', 'nb_evenements', 'ton_moyen', 'goldstein_moyen', 'nb_articles', 'pct_conflits']
                 semaine    source_type          periode  nb_evenements  ton_moyen  goldstein_moyen  nb_articles  pct_conflits
0  2025-05-26/2025-06-01  international  Periode_normale            199  -0.218281          1.01407         1208         23.12
1  2025-05-26/2025-06-01       regional  Periode_normale              5  -1.520299         -0.52000           21         40.00
2  2025-06-02/2025-06-08  international  Periode_normale            529  -2.352793         -0.02155         2991         31.38

=== local ===
['date', 'source', 'type', 'titre', 'resume', 'url', 'texte_analyse', 'theme_principal', 'tags